In [ ]:
from astroquery.mast import Mast, Observations
from astropy.io import fits
from astropy import table as tb
from astropy.time import Time

import numpy as np
from tqdm.notebook import tqdm

from glob import glob

directory = '/Volumes/hdd_1/jwst_data/prod'
dirstar = directory + '/mastDownload/JWST/**/'

In [ ]:
obs    = Observations.query_criteria(
         obs_collection = 'JWST',
         instrument_name = 'MIRI/CORON',
         calib_level = '3',
         filters = '*;4QPM*'
)
filters = {}
for filt in set(obs['filters']):
    filters[filt] = obs[ obs['filters'] == filt]
objects = []
for obj in set(obs['target_name']):
    if np.all([obj in filters[filt]['target_name'] for filt in filters]):
        objects.append(obj)

In [ ]:
for obj in tqdm(objects):
    obs = Observations.query_criteria(
          obs_collection = 'JWST',
          instrument_name = 'MIRI/CORON',
          calib_level = '3',
          target_name = obj,
          filters = '*;4QPM*'
    )
    if len(obs) == 0:
        print('nothing for ' + obj)
        continue
    t = [Observations.get_product_list(o) for o in obs]
    files = tb.unique(tb.vstack(t), keys = 'productFilename')
    print(f'-----{len(files)}------')
    manifest = Observations.download_products(
        files,
        curl_flag=False,
        calib_level = [3],
        download_dir = directory
    )

In [ ]:
objects